<a href="https://colab.research.google.com/github/billyccw/ARIN5101-Project/blob/main/ARIN5101_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Team Information**
Group:

Member 1:

Member 2:

Member 3:

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from google.colab import drive
import os

# set data sharepoint
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/ARIN5101/', exist_ok=True)

# seed setting for reproducibility
seed = 42
random.seed(seed)
rng = np.random.default_rng(seed)
torch.manual_seed(seed)

# GPU setting
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Mounted at /content/drive
cuda


In [ ]:
'''
Helper functions
'''

# this function trains and returns a model, it also saves the resulting training/validation loss and accuracy in csv file
# name param is to set name of resulting csv file. Better use model name
def train(model, loss_fn, optimizer, train_loader, val_loader, name, epochs=20):
  model_loss = {"train": [], "val": []} # each list store loss of each epoch in training and validation respectively
  model_acc = {"train": [], "val": []}
  for epoch in range(epochs):
      train_loss, train_correct, train_total, val_loss, val_correct, val_total = 0, 0, 0, 0, 0, 0
      model.train()
      for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        p = model(X_batch)
        loss = loss_fn(p, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # performance calculation
        train_loss += loss.item() * y_batch.shape[0]
        y_hat = p.argmax(1)
        train_correct += (y_hat == y_batch).sum().item()
        train_total += y_batch.shape[0]
      model_loss["train"].append(train_loss / train_total)
      model_acc["train"].append(train_correct / train_total)
      print(f"Epoch {epoch+1}: Training loss = {train_loss / train_total}, accuracy = {train_correct / train_total}")

      # validation
      model.eval()
      with torch.no_grad():
        for X_batch, y_batch in val_loader:
          X_batch, y_batch = X_batch.to(device), y_batch.to(device)
          p = model(X_batch)
          loss = loss_fn(p, y_batch)
          # performance calculation
          val_loss += loss.item() * y_batch.shape[0]
          y_hat = p.argmax(1)
          val_correct += (y_hat == y_batch).sum().item()
          val_total += y_batch.shape[0]
      model_loss["val"].append(val_loss / val_total)
      model_acc["val"].append(val_correct / val_total)
      print(f"Epoch {epoch+1}: Validation loss = {val_loss / val_total}, accuracy = {val_correct / val_total}")

  # write loss and acc to csv file for future visualization
  df = pd.DataFrame({"train_loss": model_loss["train"], "train_acc": model_acc["train"], "val_loss": model_loss["val"], "val_acc": model_acc["val"]})
  os.makedirs('/content/drive/MyDrive/ARIN5101/', exist_ok=True)
  df.to_csv(f'/content/drive/MyDrive/ARIN5101/{name}_training_performance.csv', index=False, encoding="utf-8-sig")
  return model

# This function predicts on validation set for future confusion matrix construction, also saves resulting predictions and ground-truths to csv file
# name param is to set name of resulting csv file. Better use model name
def validate(model, val_loader, name):
  model.eval()
  all_preds = []
  all_labels = []
  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      X_batch = X_batch.to(device)
      preds = model(X_batch).argmax(dim=1).cpu().numpy()
      all_preds.extend(preds)
      all_labels.extend(y_batch.numpy())
  df_pred = pd.DataFrame({
          "prediction": all_preds,
          "actual": all_labels
  })
  os.makedirs('/content/drive/MyDrive/ARIN5101/', exist_ok=True)
  df_pred.to_csv(f'/content/drive/MyDrive/ARIN5101/{name}_predictions.csv', index=False, encoding="utf-8-sig")
  print(f"Saved {len(df_pred)} predictions to '/content/drive/MyDrive/ARIN5101/{name}_predictions.csv'")



# **Dataset Loading**

In [ ]:
# Load the dataset and do some preprocessing

#### Dataset description
• Basic description, distributions and samples

# **Dataset Splitting**

In [ ]:
# Split dataset into training and validation sets by loader

#### Training setup and validation strategy
• Dataset splitting strategy

# **Baseline CNN Architecture**

In [ ]:
# Input param explaination
"""
in_channel: number of channel of input image
in_height: height of input image
num_class: number of class in dataset
conv_layer_depth: Convolution layer depth -> need to be tuned
hidden_layer_size: Hidden layer size -> need to be tuned
extra_conv: bool value identifying whether a third convolutional layer should be added
"""
class baselineCNN(nn.Module):
  def __init__(self, in_channel, in_height, num_class, conv_layer_depth=32, hidden_layer_size=128, extra_conv: bool = False):
    super().__init__()
    self.extra_conv = extra_conv
    self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels = conv_layer_depth, kernel_size=3, padding=1)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(2)
    self.conv2 = nn.Conv2d(in_channels=conv_layer_depth, out_channels = conv_layer_depth*2, kernel_size=3, padding=1)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(2)
    if extra_conv:
      self.conv3 = nn.Conv2d(in_channels=conv_layer_depth*2, out_channels = conv_layer_depth*3, kernel_size=3, padding=1)
      self.relu3 = nn.ReLU()
      self.pool3 = nn.MaxPool2d(2)
    self.flatten = nn.Flatten()
    final_img_size = in_height // (2**(2+extra_conv))
    self.linear1 = nn.Linear(final_img_size * final_img_size * conv_layer_depth*(2+extra_conv), hidden_layer_size)
    self.relu_fc = nn.ReLU()
    self.linear2 = nn.Linear(hidden_layer_size, num_class)

  def forward(self, x):
    x = self.pool1(self.relu1(self.conv1(x)))
    x = self.pool2(self.relu2(self.conv2(x)))
    if self.extra_conv:
      x = self.pool3(self.relu3(self.conv3(x)))
    x = self.flatten(x)
    x = self.relu_fc(self.linear1(x))
    x = self.linear2(x)
    return x

In [ ]:
# Train and validation
baseCNN_model = baselineCNN(in_channel, in_height, num_class).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseCNN_model.parameters(), lr=1e-3)

baseCNN_model = train(baseCNN_model, loss_fn, optimizer, train_loader, val_loader, "baselineCNN")

# After training, do predictions on validation set for future confusion matrix construction
validate(baseCNN_model, val_loader, "baselineCNN")


#### Basic illustration

#### Performance Analyse
• loss, accuracy, confusion matrix and other relevant metrics

# **Tuning hyperparam in baseline**

In [ ]:
# Not Implemented Yet

#### Basic illustration

#### Performance Analyse
• Comparisons among the combinations using relevant metrics

# **CNN model with improvement A**

In [ ]:
# Not Implemented Yet

#### Basic illustration

#### Performance Analyse
• Comparisons among the combinations using relevant metrics

# **CNN model with improvement B**

In [ ]:
# Not Implemented Yet

#### Basic illustration

#### Performance Analyse
• Comparisons among the combinations using relevant metrics

# **CNN model with improvements A and B**

In [ ]:
# Not Implemented Yet

#### Basic illustration

#### Performance Analyse
• Comparisons among the combinations using relevant metrics

# **Final model summary and reflections**

In [ ]:
# visualize performance of all models

• Which one is the best performing model and why

• Higher accuracy does not necessarily indicate a better model